# CVM DFP → Markdown + DoclingDocument (Docling, GPU)

Converts the CVM DFP PDFs, accelerated on a T4 GPU. For each filing this writes:

- `<year>/<file>.md` -- the Markdown export (for humans / quick inspection).
- `<year>/<file>.docling.json` -- the full `DoclingDocument`, including real page numbers and native structure. `backend/ingest/chunking.py` reads *this* (not the Markdown -- plain Markdown text has no page numbers to recover), via its own heading-aware chunker, not Docling's `HybridChunker` -- `HybridChunker`'s heading-trail tracking produced worse section labels on this corpus (confirmed by inspecting its output), so page numbers are threaded through as a separate step instead. See that module's docstring for details.

**Before running:** `Runtime` → `Change runtime type` → `T4 GPU`.

**Reads from and writes to Google Drive**, not the Colab VM's local disk — the VM is ephemeral and a crash (e.g. "session crashed after using all available RAM", which *will* happen occasionally on the free tier) wipes anything only on local disk. Writing to Drive means every filing that finishes converting is durable immediately, exactly like the local CPU script, and a crash costs you at most the one filing in flight, not the whole run. Re-running after a crash just reconnects to Drive and resumes (already-converted filings -- meaning both `.md` and `.docling.json` exist -- are skipped).

One-time setup: upload `downloads.zip` (zip of the local `data/downloads/` folder) when prompted the *first* time you run this notebook. On every later run — including after a crash — it's already in Drive, so that step is skipped automatically.

**If you previously hit `nvrtc: error: failed to open libnvrtc-builtins.so.13.0`**: Colab's own *preinstalled* torch is already a CUDA-13 build (`torch==2.11.0+cu130` as of writing) that needs this exact file to JIT-compile any CUDA kernel, but the image doesn't ship it. Installing `pip install nvidia-cuda-nvrtc==13.0.*` (the first cell does this) puts the file on disk (under `nvidia/cu13/lib/`), but that alone is *not* enough -- it's not a location torch's own preload hook knows to add to the dynamic loader's search path, so the pip install alone still leaves the error in place. The first cell now also explicitly adds that file's directory to `LD_LIBRARY_PATH` and preloads it with `ctypes.CDLL(..., mode=ctypes.RTLD_GLOBAL)` before any conversion runs -- **this part is confirmed fixed**, live-tested with zero NVRTC errors across multiple runs, versus every prior attempt failing within 10-50 seconds. If you still hit the NVRTC error after that, `Runtime → Disconnect and delete runtime` and try again from a clean one.

**Separate, still-unresolved issue: "session crashed after using all available RAM."** Even past the NVRTC fix above, real conversion on the free tier has crashed from system RAM exhaustion within ~10-12 minutes on the very first (smallest) file, despite already running `TABLE_MODE = TableFormerMode.FAST`, `do_ocr = False`, and `layout_batch_size = table_batch_size = 1` -- the exact settings that previously prevented this. The current best guess is that Colab's newer preinstalled CUDA-13 torch build simply has a larger baseline memory footprint than whatever image this pipeline was last verified against, but this is unconfirmed. Options if you hit this: a Colab Pro / High-RAM runtime, further cutting `PdfPipelineOptions.images_scale` or disabling table structure entirely to test, or falling back to the local CPU script (`data/convert_to_markdown.py`), which has never hit this failure mode (just much slower).

In [ ]:
# Colab's preinstalled torch is already a CUDA-13 build (torch==X.Y.Z+cu130),
# but the image doesn't ship the matching NVRTC runtime library, so any
# JIT-compiled CUDA kernel (docling's RT-DETR-v2 layout model hits one) fails
# with "nvrtc: error: failed to open libnvrtc-builtins.so.13.0". Installing
# nvidia-cuda-nvrtc (unsuffixed; the old nvidia-cuda-nvrtc-cu13 name is a
# deprecated 0.0.1 placeholder, never a real build) puts the missing file on
# disk under nvidia/cu13/lib/ -- not a path torch's own preload hook knows
# about -- so it still needs to be added to LD_LIBRARY_PATH and preloaded
# explicitly. Only that one exact file is targeted (not a broad glob under
# /usr) to avoid also loading the VM's separate, much larger full CUDA
# toolkit copies of the same-named file for no reason.
!python -c "import torch; print(f'torch=={torch.__version__}')" > /tmp/torch_constraint.txt
!cat /tmp/torch_constraint.txt
!python -c "import torch; print(torch.version.cuda)" > /tmp/cuda_version.txt
!cat /tmp/cuda_version.txt
!pip install -q "nvidia-cuda-nvrtc==$(cat /tmp/cuda_version.txt).*"
!pip install -q docling -c /tmp/torch_constraint.txt
!pip install -q "requests==2.32.4"  # keep in sync with what google-colab itself pins, to avoid a resolver conflict

import ctypes
import glob
import os

nvrtc_libs = sorted(glob.glob("/usr/local/lib/python3*/dist-packages/nvidia/cu13/lib/libnvrtc-builtins.so.13.0"))
print("Found:", nvrtc_libs)
lib_dirs = sorted({os.path.dirname(p) for p in nvrtc_libs})
os.environ["LD_LIBRARY_PATH"] = ":".join([*lib_dirs, os.environ.get("LD_LIBRARY_PATH", "")])
print("LD_LIBRARY_PATH:", os.environ["LD_LIBRARY_PATH"])
preload_results = [(p, (lambda: (ctypes.CDLL(p, mode=ctypes.RTLD_GLOBAL), "ok")[1])()) for p in nvrtc_libs]
print("Preloaded:", preload_results)

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

If `CUDA available` printed `False` above, go to `Runtime` → `Change runtime type` and select a `T4 GPU`, then re-run from the top.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/cvm-copilot")
DOWNLOADS_DIR = DRIVE_ROOT / "downloads"
MARKDOWN_DIR = DRIVE_ROOT / "markdown"
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
# One-time only: skipped automatically once downloads/ already exists in Drive
# (including on every run after the first, and after a crash-and-resume).
if not (DOWNLOADS_DIR / "manifest.json").exists():
    from google.colab import files
    print("Select downloads.zip")
    uploaded = files.upload()
    !unzip -q downloads.zip -d "{DRIVE_ROOT}"
else:
    print("downloads/ already in Drive, skipping upload")

In [ ]:
import gc
import json
import logging
import time
from datetime import UTC, datetime
from importlib.metadata import version

from docling.datamodel.base_models import ConversionStatus, InputFormat
from docling.datamodel.pipeline_options import (
    AcceleratorDevice,
    AcceleratorOptions,
    PdfPipelineOptions,
    TableFormerMode,
)
from docling.document_converter import DocumentConverter, PdfFormatOption

# FAST, not ACCURATE: ACCURATE crashed a free-tier Colab runtime by exhausting
# system RAM partway through the batch. GPU still makes FAST much faster than
# the local CPU run. If you want to try ACCURATE anyway (e.g. on Colab Pro's
# High-RAM runtime, or processing a handful of filings at a time), change this
# and watch RAM in the Colab resource meter as you go.
TABLE_MODE = TableFormerMode.FAST

# Some failures (e.g. a CUDA kernel compile error) embed a huge generated
# source dump in the exception text -- cap it so one failure can't blow the
# conversion log up to tens of MB.
MAX_ERROR_MESSAGE_LENGTH = 2000

logging.basicConfig(level=logging.INFO, format="%(message)s")
logger = logging.getLogger(__name__)


def build_converter() -> DocumentConverter:
    pipeline_options = PdfPipelineOptions(do_table_structure=True)
    # CVM DFPs are digitally-native PDFs, not scans -- OCR was running (and
    # loading its own models) on pages that never needed it, for no benefit.
    pipeline_options.do_ocr = False
    # Smaller batches trade some speed for a much lower peak memory footprint,
    # which is the actual constraint on a free-tier Colab runtime.
    pipeline_options.layout_batch_size = 1
    pipeline_options.table_batch_size = 1
    pipeline_options.table_structure_options.mode = TABLE_MODE
    pipeline_options.accelerator_options = AcceleratorOptions(device=AcceleratorDevice.CUDA)
    return DocumentConverter(format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)})


def load_manifest() -> dict:
    return json.loads((DOWNLOADS_DIR / "manifest.json").read_text(encoding="utf-8"))


def markdown_path(local_path: str) -> Path:
    return MARKDOWN_DIR / Path(local_path).with_suffix(".md")


def docling_json_path(local_path: str) -> Path:
    # The DoclingDocument itself -- not just its Markdown export -- so page
    # numbers and native structure survive for ingest/chunking.py to use, via
    # its own heading-aware chunker (deliberately not Docling's HybridChunker --
    # see backend/ingest/chunking.py's docstring for why).
    return MARKDOWN_DIR / Path(local_path).with_suffix(".docling.json")


def is_done(local_path: str) -> bool:
    return markdown_path(local_path).exists() and docling_json_path(local_path).exists()


def write_markdown_manifest(manifest: dict) -> None:
    converted_filings = [
        {**filing, "local_path": str(Path(filing["local_path"]).with_suffix(".md"))}
        for filing in manifest["filings"]
        if is_done(filing["local_path"])
    ]
    converted_manifest = {**manifest, "filings": converted_filings}
    MARKDOWN_DIR.mkdir(parents=True, exist_ok=True)
    (MARKDOWN_DIR / "manifest.json").write_text(
        json.dumps(converted_manifest, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
    )
    logger.info("Wrote markdown/manifest.json with %d/%d filing(s)", len(converted_filings), len(manifest["filings"]))


def write_conversion_log(run_started_at: datetime, filing_runs: list[dict], *, total_filings: int, skipped: int) -> None:
    succeeded = sum(1 for run in filing_runs if run["status"] == ConversionStatus.SUCCESS.value)
    partial = sum(1 for run in filing_runs if run["status"] == ConversionStatus.PARTIAL_SUCCESS.value)
    failed = len(filing_runs) - succeeded - partial
    log = {
        "run_started_at": run_started_at.isoformat(),
        "docling_version": version("docling"),
        "pipeline_options": {
            "do_table_structure": True,
            "do_ocr": False,
            "table_structure_mode": TABLE_MODE.value,
            "accelerator_device": "cuda",
        },
        "total_filings": total_filings,
        "skipped_already_converted": skipped,
        "attempted": len(filing_runs),
        "succeeded": succeeded,
        "partial_success": partial,
        "failed": failed,
        "total_duration_seconds": round(sum(run["duration_seconds"] for run in filing_runs), 1),
        "filings": filing_runs,
    }
    MARKDOWN_DIR.mkdir(parents=True, exist_ok=True)
    (MARKDOWN_DIR / "conversion_log.json").write_text(json.dumps(log, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    logger.info("Wrote markdown/conversion_log.json")


def main() -> None:
    run_started_at = datetime.now(UTC)
    manifest = load_manifest()

    pending = [filing for filing in manifest["filings"] if not is_done(filing["local_path"])]
    # Smallest files first: build up progress on easy filings and isolate the
    # handful of very large ones (e.g. a bank's DFP can be 4x the typical size)
    # as the last, most-likely-to-need-attention items instead of blocking on
    # one of them immediately.
    pending.sort(key=lambda filing: (DOWNLOADS_DIR / filing["local_path"]).stat().st_size)
    skipped = len(manifest["filings"]) - len(pending)
    if skipped:
        logger.info("Skipping %d already-converted filing(s)", skipped)

    filing_runs = []
    if pending:
        converter = build_converter()
        checkpoint = time.monotonic()
        started_at = checkpoint

        # One file at a time (not convert_all()) so the page images and
        # torch/CUDA buffers for a finished document can be freed before the
        # next one starts, instead of accumulating across the whole batch.
        for filing in pending:
            pdf_path = DOWNLOADS_DIR / filing["local_path"]
            result = converter.convert(pdf_path, raises_on_error=False)

            now = time.monotonic()
            duration_seconds = now - checkpoint
            checkpoint = now

            errors = [error.error_message[:MAX_ERROR_MESSAGE_LENGTH] for error in result.errors]

            if result.status in (ConversionStatus.SUCCESS, ConversionStatus.PARTIAL_SUCCESS):
                out_path = markdown_path(filing["local_path"])
                out_path.parent.mkdir(parents=True, exist_ok=True)
                out_path.write_text(result.document.export_to_markdown(), encoding="utf-8")
                result.document.save_as_json(docling_json_path(filing["local_path"]))
                logger.info("Converted %s (%.1fs)", filing["local_path"], duration_seconds)
            else:
                logger.error("Failed to convert %s (%.1fs)", filing["local_path"], duration_seconds)
                for message in errors:
                    logger.error("  %s", message)

            filing_runs.append(
                {
                    "ticker": filing["ticker"],
                    "fiscal_year": filing["fiscal_year"],
                    "local_path": filing["local_path"],
                    "status": result.status.value,
                    "duration_seconds": round(duration_seconds, 1),
                    "errors": errors,
                }
            )

            # Write the manifest/log after every file, not just at the end, so
            # a crash mid-batch still leaves a correct record of what finished.
            write_markdown_manifest(manifest)
            write_conversion_log(run_started_at, filing_runs, total_filings=len(manifest["filings"]), skipped=skipped)

            del result
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        logger.info("Converted %d filing(s) in %.1fs total", len(pending), time.monotonic() - started_at)
    else:
        write_markdown_manifest(manifest)
        write_conversion_log(run_started_at, filing_runs, total_filings=len(manifest["filings"]), skipped=skipped)


main()

Done — results are already durable in Drive at `MyDrive/cvm-copilot/markdown/`. If the runtime crashes partway through, just re-run this notebook from the top; the upload step is skipped (already in Drive) and the conversion loop picks up from whichever filing it stopped on.

To pull the results into the local repo: open `MyDrive/cvm-copilot/markdown/` in Drive's web UI and download it (as a zip), or install Google Drive for desktop and sync it locally, then copy its contents into `data/markdown/` — same `<year>/<file>.md` layout, so it merges cleanly with (and is treated as already-converted by) the local CPU run.

Optional: zip and download directly from this notebook instead, if you'd rather not deal with Drive's UI:

In [ ]:
!zip -qr /content/markdown.zip "{MARKDOWN_DIR}"
from google.colab import files
files.download("/content/markdown.zip")